# RetailPulse -- Cross-Validation & Model Refinement

**Objective:** Implement time-series cross-validation and walk-forward validation for robust model evaluation.

In [1]:
import os, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
def save_fig(fig, name):
    fig.savefig(os.path.join(FIGURES_DIR, name), dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig); print(f"Saved: {name}")


## Walk-Forward Validation

Expand the training window progressively, always predicting the next 30 days.

In [2]:
df = pd.read_csv(os.path.join(PROCESSED_DIR, "prophet_ready.csv"), parse_dates=["ds"])
df = df.sort_values("ds").reset_index(drop=True)
print(f"Total data: {len(df)} days")

HORIZON = 30
MIN_TRAIN = 180
fold_results = []
fold_num = 0

for start_test in range(MIN_TRAIN, len(df) - HORIZON, HORIZON):
    fold_num += 1
    train = df.iloc[:start_test]
    test = df.iloc[start_test:start_test + HORIZON]
    if len(test) < 10:
        break
    m = Prophet(weekly_seasonality=True, daily_seasonality=False, yearly_seasonality=False)
    m.add_seasonality(name="monthly", period=30.5, fourier_order=5)
    m.fit(train)
    future = m.make_future_dataframe(periods=len(test))
    fc = m.predict(future)
    pred = fc[fc["ds"].isin(test["ds"])]["yhat"].values
    actual = test["y"].values[:len(pred)]
    non_zero = actual != 0
    mape = np.mean(np.abs((actual[non_zero] - pred[non_zero]) / actual[non_zero])) * 100 if non_zero.sum() > 0 else np.nan
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    fold_results.append({"Fold": fold_num, "Train Days": len(train), "Test Days": len(test),
                          "MAPE (%)": round(mape, 2), "MAE": round(mae, 2), "RMSE": round(rmse, 2)})
    print(f"  Fold {fold_num}: train={len(train)}, test={len(test)}, MAPE={mape:.2f}%")

cv_df = pd.DataFrame(fold_results)
print(f"\nAvg MAPE across folds: {cv_df['MAPE (%)'].mean():.2f}% +/- {cv_df['MAPE (%)'].std():.2f}%")


16:17:19 - cmdstanpy - INFO - Chain [1] start processing


Total data: 739 days


16:17:19 - cmdstanpy - INFO - Chain [1] done processing


16:17:20 - cmdstanpy - INFO - Chain [1] start processing


16:17:20 - cmdstanpy - INFO - Chain [1] done processing


...

  Fold 16: train=630, test=30, MAPE=35.44%


16:17:24 - cmdstanpy - INFO - Chain [1] start processing


16:17:24 - cmdstanpy - INFO - Chain [1] done processing


  Fold 17: train=660, test=30, MAPE=33.08%


  Fold 18: train=690, test=30, MAPE=24.36%

Avg MAPE across folds: 42.97% +/- 31.41%


In [3]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
axes[0].bar(cv_df["Fold"], cv_df["MAPE (%)"], color="#3498db", alpha=0.8)
axes[0].axhline(cv_df["MAPE (%)"].mean(), color="#e74c3c", linestyle="--", label=f'Mean={cv_df["MAPE (%)"].mean():.1f}%')
axes[0].set_xlabel("Fold"); axes[0].set_ylabel("MAPE (%)"); axes[0].set_title("MAPE per Fold"); axes[0].legend()
axes[1].bar(cv_df["Fold"], cv_df["MAE"], color="#27ae60", alpha=0.8)
axes[1].set_xlabel("Fold"); axes[1].set_ylabel("MAE"); axes[1].set_title("MAE per Fold")
axes[2].bar(cv_df["Fold"], cv_df["RMSE"], color="#9b59b6", alpha=0.8)
axes[2].set_xlabel("Fold"); axes[2].set_ylabel("RMSE"); axes[2].set_title("RMSE per Fold")
fig.suptitle("Walk-Forward Cross-Validation Results", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout(); save_fig(fig, "39_cv_results.png"); plt.show()


Saved: 39_cv_results.png


In [4]:
cv_df.to_csv(os.path.join(PROCESSED_DIR, "cv_results.csv"), index=False)
print("Saved: cv_results.csv")
print(f"\nMODEL REFINEMENT COMPLETE")
print(f"Walk-forward CV MAPE: {cv_df['MAPE (%)'].mean():.2f}% +/- {cv_df['MAPE (%)'].std():.2f}%")


Saved: cv_results.csv

MODEL REFINEMENT COMPLETE
Walk-forward CV MAPE: 42.97% +/- 31.41%
